# Wave stochastic lifting with a U-Net

Train a one-step stochastic lifting model with a 3-frame Markov context, then sample 128 rollouts from the shared initial wave frame.

In [ ]:
import functools
import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax.training import train_state
from tqdm.auto import trange

from sl.plot import plot_grid_movie as imshow_movie_grid
from sl.unet import UNet
from sl.wave import get_wave_data

key = jax.random.PRNGKey(0)
n_train, n_rollout = 128, 128
n_t, n_x_solve, sub_x = 32, 128, 2
history, label_dim = 3, 64
batch_size, train_steps, lr = 64, 2_000, 1e-4

In [ ]:
key, data_key, label_key = jax.random.split(key, 3)
trajectories = get_wave_data(
    key=data_key,
    n_samples=n_train,
    n_x=n_x_solve,
    n_t=n_t,
    sigma=1,
    sub_x=sub_x,
    batch_size=64
).astype(np.float32)
trajectories /= np.abs(trajectories).max()
trajectories.shape

In [ ]:
def make_transitions(xs, history=3):
    n, t, h, w, c = xs.shape
    padded = jnp.concatenate([jnp.repeat(xs[:, :1], history - 1, axis=1), xs], axis=1)
    contexts = jnp.stack([padded[:, i : i + history] for i in range(t - 1)], axis=1)
    contexts = contexts.transpose(0, 1, 3, 4, 2, 5).reshape(n * (t - 1), h, w, history * c)
    targets = xs[:, 1:].reshape(n * (t - 1), h, w, c)
    return contexts, targets

contexts, targets = make_transitions(jnp.asarray(trajectories), history)
labels = jax.random.normal(label_key, (contexts.shape[0], label_dim), dtype=jnp.float32)
contexts.shape, labels.shape, targets.shape

In [ ]:
model = UNet(
    out_channels=1,
    emb_features=[256, 256],
    feature_depths=[32, 64, 128],
    num_res_blocks=1,
    num_middle_res_blocks=1,
    label_in="conditional",
)

key, init_key = jax.random.split(key)
params = model.init(init_key, contexts[:1], labels[:1], None, train=True)["params"]
schedule = optax.cosine_decay_schedule(lr, train_steps)
state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=optax.adamw(schedule))

@jax.jit
def train_step(state, x, xi, y):
    def loss_fn(params):
        pred = state.apply_fn({"params": params}, x, xi, None, train=True)
        return jnp.mean((pred - y) ** 2)

    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    return state.apply_gradients(grads=grads), loss

def sample_batch(key):
    idx = jax.random.randint(key, (batch_size,), 0, contexts.shape[0])
    return contexts[idx], labels[idx], targets[idx]

In [ ]:
losses = []
pbar = trange(train_steps)
for step in pbar:
    key, batch_key = jax.random.split(key)
    state, loss = train_step(state, *sample_batch(batch_key))
    if step % 20 == 0:
        losses.append(float(loss))
        pbar.set_postfix(loss=f"{losses[-1]:.3e}")

In [ ]:
@functools.partial(jax.jit, static_argnames=("steps",))
def rollout(params, x0, key, steps):
    def step(context, key):
        xi = jax.random.normal(key, (context.shape[0], label_dim), dtype=context.dtype)
        x_next = model.apply({"params": params}, context, xi, None, train=False)
        context = jnp.concatenate([context[..., 1:], x_next], axis=-1)
        return context, x_next

    context0 = jnp.repeat(x0, history, axis=-1)
    _, xs = jax.lax.scan(step, context0, jax.random.split(key, steps))
    xs = jnp.swapaxes(xs, 0, 1)
    return jnp.concatenate([x0[:, None], xs], axis=1)

key, rollout_key = jax.random.split(key)
x0 = jnp.repeat(jnp.asarray(trajectories[:1, 0]), n_rollout, axis=0)
rollouts = np.asarray(rollout(state.params, x0, rollout_key, steps=n_t - 1))
rollouts.shape

In [ ]:
movies = rollouts[:9, ..., 0]
v = float(np.abs(movies).max())
imshow_movie_grid(
    movies,
    grid_height=3,
    grid_width=3,
    fig_size=(7, 7),
    cmap="RdBu_r",
    c_norm=(-v, v),
    live_cbar=False,
    frames=n_t,
    interval=120,
)

## Wasserstein metric plots

Compare the true wave ensemble against the generated rollouts through integrated mass and boundary crossing time distributions.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from sl.metric import crossing_time_wasserstein_2, mass_wasserstein_2

true_eval = trajectories
pred_eval = rollouts
t_grid = np.linspace(0.0, 1.0, true_eval.shape[1])

true_mass, pred_mass, mass_w2_t, mass_w2 = mass_wasserstein_2(true_eval, pred_eval)

crossing_threshold = 0.05
true_tau, pred_tau, crossing_w2 = crossing_time_wasserstein_2(
    true_eval,
    pred_eval,
    threshold=crossing_threshold,
)

print(f"Mass W2 mean: {mass_w2:.4f}")
print(f"Boundary crossing-time W2: {crossing_w2:.4f}")


In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
true_color = "#2f5d8c"
pred_color = "#d95f02"
selected = np.unique(np.linspace(0, true_eval.shape[1] - 1, 4, dtype=int))
time_colors = sns.color_palette("viridis", len(selected))

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
fig.suptitle("Wave rollout distribution metrics", fontsize=16, fontweight="semibold")

ax = axes[0, 0]
ax.plot(t_grid, mass_w2_t, color="#1b4d3e", linewidth=2.5)
ax.fill_between(t_grid, 0.0, mass_w2_t, color="#1b4d3e", alpha=0.14)
ax.axhline(mass_w2, color="#1b4d3e", linestyle="--", linewidth=1.4, alpha=0.8)
ax.text(
    0.98,
    0.94,
    f"mean = {mass_w2:.3f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=11,
    color="#1b4d3e",
)
ax.set_title("Mass Wasserstein distance over time")
ax.set_xlabel("normalized time")
ax.set_ylabel("W2")

ax = axes[0, 1]
for idx, color in zip(selected, time_colors):
    sns.ecdfplot(
        x=true_mass[:, idx],
        ax=ax,
        color=color,
        linewidth=2.0,
        label=f"true t={t_grid[idx]:.2f}",
    )
    sns.ecdfplot(
        x=pred_mass[:, idx],
        ax=ax,
        color=color,
        linewidth=2.0,
        linestyle="--",
        label=f"pred t={t_grid[idx]:.2f}",
    )
ax.set_title("Integrated mass distributions")
ax.set_xlabel("mean absolute field value")
ax.set_ylabel("empirical CDF")
ax.legend(frameon=False, fontsize=8, ncol=2)

ax = axes[1, 0]
bins = np.linspace(0.0, 1.0, min(true_eval.shape[1] + 1, 25))
ax.hist(
    true_tau,
    bins=bins,
    density=True,
    color=true_color,
    alpha=0.38,
    edgecolor=true_color,
    linewidth=1.3,
    label="true",
)
ax.hist(
    pred_tau,
    bins=bins,
    density=True,
    color=pred_color,
    alpha=0.34,
    edgecolor=pred_color,
    linewidth=1.3,
    label="pred",
)
ax.axvline(true_tau.mean(), color=true_color, linewidth=2.0)
ax.axvline(pred_tau.mean(), color=pred_color, linewidth=2.0, linestyle="--")
ax.set_title(f"Boundary crossing time, W2 = {crossing_w2:.3f}")
ax.set_xlabel(f"first boundary hit time, threshold={crossing_threshold:g}")
ax.set_ylabel("density")
ax.legend(frameon=False)

ax = axes[1, 1]
sns.ecdfplot(x=true_tau, ax=ax, color=true_color, linewidth=2.5, label="true")
sns.ecdfplot(x=pred_tau, ax=ax, color=pred_color, linewidth=2.5, linestyle="--", label="pred")
ax.set_title("Boundary crossing-time distributions")
ax.set_xlabel("normalized first hit time")
ax.set_ylabel("empirical CDF")
ax.legend(frameon=False)

for ax in axes.ravel():
    ax.spines[["top", "right"]].set_visible(False)

plt.show()
